# Projeto de assistente de pedido do Mc Donalds
Um agente conversacional em português que ajuda a montar um pedido do McDonald's. Ele entende restrições escritas livremente ("frango com baixo sódio"). Se o usuário não disser nenhuma, o agente pergunta. As respostas vêm do `menu.csv`, não da memória do modelo.

### Instalações


In [ ]:
!pip install langchain langchain_openai nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 13.9 MB/s  0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain_openai]


### Chave e modelo

In [ ]:
from getpass import getpass

OPENAI_API_KEY = getpass()

··········


In [ ]:
from langchain_openai import ChatOpenAI
modelo = ChatOpenAI(model="gpt-6-luna", api_key=OPENAI_API_KEY)

print(modelo.invoke("Responda apenas 'ok' se estiver funcionando.").content)

ok


### Carregar o corpus
Baixa o csv do GitHub

In [ ]:
import requests
import pandas as pd
from io import StringIO

URL_MENU = "https://raw.githubusercontent.com/BeatrizGiacon/Projeto-Pratico-PLN-/main/menu.csv"

resposta = requests.get(URL_MENU, timeout=30)
resposta.raise_for_status()
df_bruto = pd.read_csv(StringIO(resposta.text))

print(df_bruto.shape)
df_bruto.head()


(260, 24)


,Category,Item,Serving Size,Calories,Calories from Fat,Total Fat,Total Fat (% Daily Value),Saturated Fat,Saturated Fat (% Daily Value),Trans Fat,...,Carbohydrates,Carbohydrates (% Daily Value),Dietary Fiber,Dietary Fiber (% Daily Value),Sugars,Protein,Vitamin A (% Daily Value),Vitamin C (% Daily Value),Calcium (% Daily Value),Iron (% Daily Value)
0,Breakfast,Egg McMuffin,4.8 oz (136 g),300,120,13.0,20,5.0,25,0.0,...,31,10,4,17,3,17,10,0,25,15
1,Breakfast,Egg White Delight,4.8 oz (135 g),250,70,8.0,12,3.0,15,0.0,...,30,10,4,17,3,18,6,0,25,8
2,Breakfast,Sausage McMuffin,3.9 oz (111 g),370,200,23.0,35,8.0,42,0.0,...,29,10,4,17,2,14,8,0,25,10
3,Breakfast,Sausage McMuffin with Egg,5.7 oz (161 g),450,250,28.0,43,10.0,52,0.0,...,30,10,4,17,2,21,15,0,30,15
4,Breakfast,Sausage McMuffin with Egg Whites,5.7 oz (161 g),400,210,23.0,35,8.0,42,0.0,...,30,10,4,17,2,21,6,0,25,10


In [ ]:
renomear = {
    "Category": "categoria", "Item": "item", "Serving Size": "porcao_texto",
    "Calories": "calorias", "Total Fat": "gordura_g", "Saturated Fat": "gordura_sat_g",
    "Cholesterol": "colesterol_mg", "Sodium": "sodio_mg", "Carbohydrates": "carbo_g",
    "Sugars": "acucar_g", "Protein": "proteina_g", "Calcium (% Daily Value)": "calcio_vd_pct",
}
df = df_bruto[list(renomear)].rename(columns=renomear).copy()

# Limpa símbolos e espaços dos textos
df["item"] = df["item"].str.replace(r"[®™]", "", regex=True).str.strip()
df["categoria"] = df["categoria"].str.strip()

# Peso da porção em gramas (bebidas medidas em ml ficam sem valor)
df["porcao_g"] = df["porcao_texto"].str.extract(r"\((\d+(?:\.\d+)?) g\)").astype(float)

print("Itens com nome repetido:", df["item"].duplicated().sum())
df.head()

Itens com nome repetido: 0


,categoria,item,porcao_texto,calorias,gordura_g,gordura_sat_g,colesterol_mg,sodio_mg,carbo_g,acucar_g,proteina_g,calcio_vd_pct,porcao_g
0,Breakfast,Egg McMuffin,4.8 oz (136 g),300,13.0,5.0,260,750,31,3,17,25,136.0
1,Breakfast,Egg White Delight,4.8 oz (135 g),250,8.0,3.0,25,770,30,3,18,25,135.0
2,Breakfast,Sausage McMuffin,3.9 oz (111 g),370,23.0,8.0,45,780,29,2,14,25,111.0
3,Breakfast,Sausage McMuffin with Egg,5.7 oz (161 g),450,28.0,10.0,285,860,30,2,21,30,161.0
4,Breakfast,Sausage McMuffin with Egg Whites,5.7 oz (161 g),400,23.0,8.0,50,880,30,2,21,25,161.0


### Preparar o texto do menu

Os nomes dos itens ("Big Mac", "Chicken McNuggets") são o único texto do corpus. Este passo segue o tratamento de:

- Normalização (minúsculas, sem acentos) e tokenização.
- Remover palavras vazias como "the" e "of" (stopwords) e faz lematização.
- Criar etiquetas úteis.


In [ ]:
import re, unicodedata, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords"); nltk.download("wordnet"); nltk.download("omw-1.4")
stop_en = set(stopwords.words("english"))
lematizador = WordNetLemmatizer()

def preprocessar_nome(texto):
    """Normaliza, tokeniza, remove stopwords e lematiza um nome de item."""
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode().lower()
    tokens = re.findall(r"[a-z]+", texto)
    return [lematizador.lemmatize(t) for t in tokens if t not in stop_en and len(t) > 2]

df["tokens"] = df["item"].apply(preprocessar_nome)
df[["item", "tokens"]].head(10)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,item,tokens
0,Egg McMuffin,"[egg, mcmuffin]"
1,Egg White Delight,"[egg, white, delight]"
2,Sausage McMuffin,"[sausage, mcmuffin]"
3,Sausage McMuffin with Egg,"[sausage, mcmuffin, egg]"
4,Sausage McMuffin with Egg Whites,"[sausage, mcmuffin, egg, white]"
5,Steak & Egg McMuffin,"[steak, egg, mcmuffin]"
6,"Bacon, Egg & Cheese Biscuit (Regular Biscuit)","[bacon, egg, cheese, biscuit, regular, biscuit]"
7,"Bacon, Egg & Cheese Biscuit (Large Biscuit)","[bacon, egg, cheese, biscuit, large, biscuit]"
8,"Bacon, Egg & Cheese Biscuit with Egg Whites (R...","[bacon, egg, cheese, biscuit, egg, white, regu..."
9,"Bacon, Egg & Cheese Biscuit with Egg Whites (L...","[bacon, egg, cheese, biscuit, egg, white, larg..."


### Documentação para busca semântica
Transforma cada linha em um Document do LangChain, com um texto descritivo e metadados. É a base do RAG.

In [ ]:
from langchain_core.documents import Document

def descrever(linha):
    return (f"{linha.item} (categoria: {linha.categoria}). Calorias: {linha.calorias:.0f}; "
            f"sódio: {linha.sodio_mg:.0f} mg; gordura: {linha.gordura_g:.0f} g; "
            f"açúcar: {linha.acucar_g:.0f} g; proteína: {linha.proteina_g:.0f} g.")

documentos = [
    Document(page_content=descrever(l), metadata={"item": l.item, "categoria": l.categoria})
    for l in df.itertuples()
]

**Busca por similaridade de textos**

Usamos TF-IDF com similaridade de cosseno, sem depender de embeddings externos. A busca usa apenas linhas do CSV e é reproduzível. O ranking por relevância é separado da aplicação de restrições numéricas.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

corpus_busca = [f"{d.metadata['item']} {d.metadata['categoria']} {d.page_content}" for d in documentos]
vetorizador = TfidfVectorizer(strip_accents="unicode", lowercase=True, ngram_range=(1, 2), stop_words="english")
matriz_tfidf = vetorizador.fit_transform(corpus_busca)

def buscar_similares(consulta: str, ids_permitidos=None, k=8):
    """Retorna IDs existentes no CSV, ordenados por similaridade cosseno."""
    ids = list(range(len(df))) if ids_permitidos is None else list(ids_permitidos)
    if not ids:
        return []
    consulta_vetor = vetorizador.transform([consulta])
    pontuacoes = cosine_similarity(consulta_vetor, matriz_tfidf[ids]).ravel()
    ordem = np.argsort(-pontuacoes, kind="stable")
    return [(int(ids[j]), float(pontuacoes[j])) for j in ordem[:k]]

print("Exemplo de busca:", [(df.iloc[i]['item'], round(s, 3)) for i, s in buscar_similares('chicken nuggets', k=5)])